# Fine-tune Laya for Decision Arena

Teaches Laya to fly the Descent lander by copying the autopilot. Input: `train.jsonl` and `eval.jsonl` from `pnpm laya-dataset` (in `bridge/laya/data/`). Output: `laya-descent-v1.zip`, which goes back to the Mac and is converted to MLX there.

Before running: **Runtime → Change runtime type → A100 GPU** (L4 also works, slower). Then **Runtime → Run all**. The upload cell waits for you to pick the two files.

In [ ]:
# 1. Check the GPU and install Laya (PyTorch is preinstalled on Colab).
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q laya==0.4.17
import importlib.metadata as md, torch
print("laya", md.version("laya"), "| torch", torch.__version__, "| CUDA", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 GPU, then run again."

In [ ]:
# 2. Upload train.jsonl and eval.jsonl (click "Choose files", select both).
from google.colab import files
uploaded = files.upload()
for name in ("train.jsonl", "eval.jsonl"):
    assert name in uploaded, f"{name} was not uploaded"
    print(name, sum(1 for _ in open(name)), "rows")

In [ ]:
# 3. Download the base model (the PyTorch original of aac6fef/laya-mlx).
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
snapshot_download("convaiinnovations/laya", local_dir="laya_base")
_fix_tokenizer_config("laya_base")

In [ ]:
# 4. Fine-tune. Full model (encoder and heads); 3 epochs over ~10.9k states.
#    eval.jsonl uses different courses, so the before/after numbers are held out.
import json, time
from laya.train import TrainConfig, finetune
config = TrainConfig(
    epochs=3,
    micro_batch=16,
    grad_accum=2,      # effective batch 32
    max_len=512,       # the arena's state is ~300 tokens; matches the English checkpoint
    head_max_len=256,
    eval_data="eval.jsonl",
    seed=42,
)
started = time.time()
summary = finetune("train.jsonl", "laya_base", "laya-descent-v1", config, device="cuda")
print(f"trained in {(time.time() - started) / 60:.1f} min")

In [ ]:
# 5. Before/after on held-out courses: accuracy should rise well above the base model's ~0.33.
report = json.load(open("laya-descent-v1/train_report.json"))
print("eval:", report["eval_mode"], report["eval_items"], "items")
for when in ("before", "after"):
    r = report[when]
    print(f"{when:6}  accuracy {r['accuracy']:.3f}  loss {r['loss']:.3f}  ece {r['ece']:.3f}")
print("temperatures", summary.get("temperature"))

In [ ]:
# 6. Zip the final model (without the per-epoch copy) and download it.
!cd laya-descent-v1 && rm -rf checkpoint_latest && zip -qr ../laya-descent-v1.zip . && cd .. && ls -lh laya-descent-v1.zip
files.download("laya-descent-v1.zip")